In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks

def visualize_hpp(image_path: str,
                  smooth_size: int = 7,
                  min_peak_height_ratio: float = 0.05,
                  min_distance_px: int = 55,
                  prominence_ratio: float = 0.10,
                  save_path: str = None):
    """
    Muestra una imagen binarizada junto a su Perfil de Proyección Horizontal (HPP)
    en el eje Y, con los valles (separadores de línea) marcados.

    Parameters
    ----------
    image_path          : ruta a la imagen (se binariza con Otsu si no lo está)
    smooth_size         : ventana del filtro de suavizado (uniform_filter1d)
    min_peak_height_ratio: proporción del máximo para considerar un pico
    min_distance_px     : distancia mínima en px entre picos
    prominence_ratio    : prominencia mínima relativa al máximo
    save_path           : si se indica, guarda la figura en esa ruta
    """
    img_bgr = cv2.imread(image_path)
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    H, W = binary.shape

    ink_proj   = np.sum(binary < 128, axis=1).astype(float)   # píxeles de tinta por fila
    ink_smooth = uniform_filter1d(ink_proj, size=smooth_size)

    # ── 3. Picos (centros de líneas) → valles entre ellos ───────────────────
    max_val = ink_smooth.max()
    peaks, _ = find_peaks(
        ink_smooth,
        height     = max_val * min_peak_height_ratio,
        distance   = min_distance_px,
        prominence = max_val * prominence_ratio,
    )

    valleys = []
    if len(peaks) >= 2:
        for i in range(len(peaks) - 1):
            seg = ink_smooth[peaks[i]:peaks[i+1]]
            valleys.append(peaks[i] + int(np.argmin(seg)))

    aspect = H / W
    fig_w  = 12
    fig_h  = min(max(fig_w * aspect * 0.75, 6), 22)   # escala proporcional, capped a 22"
    fig = plt.figure(figsize=(fig_w, fig_h), dpi=120)
    fig.patch.set_facecolor('#FAFAF8')

    gs = gridspec.GridSpec(
        1, 2,
        width_ratios=[3, 1],
        wspace=0.0,
        left=0.04, right=0.97,
        top=0.93, bottom=0.06,
    )

    ax_img  = fig.add_subplot(gs[0])
    ax_hist = fig.add_subplot(gs[1], sharey=ax_img)

    # — Imagen binarizada —
    ax_img.imshow(binary, cmap='gray', aspect='auto',
                  extent=[0, W, H, 0], interpolation='nearest')
    ax_img.set_xlim(0, W)
    ax_img.set_ylim(H, 0)
    ax_img.set_xlabel('columna (px)', fontsize=9, color='#555')
    ax_img.set_ylabel('fila (px)',    fontsize=9, color='#555')
    ax_img.tick_params(labelsize=8)

    # Líneas de valle sobre la imagen
    for v in valleys:
        ax_img.axhline(v, color='#D85A30', linewidth=1.2,
                       linestyle='--', alpha=0.75)

    # — Histograma horizontal (barras crecen hacia la derecha) —
    rows = np.arange(H)
    ax_hist.fill_betweenx(rows, 0, ink_smooth,
                          color='#378ADD', alpha=0.55, linewidth=0)
    ax_hist.plot(ink_smooth, rows,
                 color='#185FA5', linewidth=0.9, alpha=0.9)

    # Líneas de valle sobre el histograma
    for v in valleys:
        ax_hist.axhline(v, color='#D85A30', linewidth=1.2,
                        linestyle='--', alpha=0.75,
                        label='valle (separador)' if v == valleys[0] else '')

    ax_hist.set_xlabel('píxeles de tinta', fontsize=9, color='#555')
    ax_hist.tick_params(axis='y', labelleft=False, length=0)
    ax_hist.tick_params(axis='x', labelsize=8)
    ax_hist.set_xlim(0, max_val * 1.08)
    ax_hist.spines['left'].set_visible(False)
    ax_hist.grid(axis='x', linestyle=':', linewidth=0.5, alpha=0.5)

    # Fondo coherente
    for ax in (ax_img, ax_hist):
        ax.set_facecolor('#FAFAF8')
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
            spine.set_color('#CCCCC0')

    # — Título y leyenda —
    fig.suptitle('Perfil de proyección horizontal (HPP)',
                 fontsize=11, fontweight='bold', color='#222', y=0.97)

    if valleys:
        ax_hist.legend(fontsize=8, loc='lower right',
                       framealpha=0.6, edgecolor='#CCCCCC')

    info = f'{len(peaks)} líneas detectadas · {len(valleys)} valles · suavizado={smooth_size}px'
    fig.text(0.5, 0.01, info, ha='center', fontsize=8, color='#888')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Guardada en: {save_path}')

    plt.show()

visualize_hpp(
    image_path  = 'ampliadas/0db780ef16d50e519302a7a0e1aa4fd3.jpg',
    smooth_size = 7,
    save_path   = 'hpp_visualization.png',
)

In [ ]:
import json
import random
import sys
from collections import defaultdict
from math import ceil, floor
from sklearn.model_selection import train_test_split

prefixs = ['Tangerine-Regular_', 'ReenieBeanie-Regular_', 'PatrickHand-Regular_', 'Parisienne-Regular_', 'Pacifico-Regular_',
          'OvertheRainbow-Regular_', 'OoohBaby-Regular_', 'MsMadi-Regular_', 'MrDafoe-Regular_', 'Meddon-Regular_',
          'Kalam-Regular_', 'IndieFlower-Regular_', 'HomemadeApple-Regular_', 'Handlee-Regular_', 'GreatVibes-Regular_',
          'DancingScript-Regular_', 'CedarvilleCursive-Regular_', 'Caveat-Regular_', 'BethEllen-Regular_', 'ArchitectsDaughter-Regular_',
          'AnnieUseYourTelescope-Regular_', 'Allura-Regular_', 'AlexBrush-Regular_', 'Mayonice_']

def split_stratified(input_path, train_ratio=0.9, val_ratio=0.1, test_ratio=0.0, seed=42):
    # assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9, "Ratios must sum to 1"

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    groups = defaultdict(list)
    for key, value in data.items():
        prefix = None
        for prefix_i in prefixs:
            if key.startswith(prefix_i):
                prefix = prefix_i
                break
        
        if prefix:
            groups[prefix].append((key, value))
        else:
            groups["otros"].append((key, value))

    train, val, test = {}, {}, {}
    random.seed(seed)

    for prefix, items in groups.items():
        random.shuffle(items)
        n = len(items)

        # n_test  = max(1, round(n * test_ratio))  if n >= 3 else 0
        n_val   = max(1, round(n * val_ratio))   if n >= 2 else 0
        n_train = n - n_val #- n_test

        # Evitar negativos si el grupo es muy pequeño
        if n_train < 0:
            n_train = 0

        train_items = items[:n_train]
        val_items   = items[n_train:n_train + n_val]
        # test_items  = items[n_train + n_val:]

        for k, v in train_items:
            train[k] = v
        for k, v in val_items:
            val[k] = v
        # for k, v in test_items:
        #     test[k] = v

    # Guardar outputs
    base = input_path.rsplit(".", 1)[0]
    for split_name, split_data in [("train", train), ("val", val), ("test", test)]:
        out_path = f"{base}_{split_name}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(split_data, f, ensure_ascii=False, indent=2)
        print(f"[{split_name}] {len(split_data)} items → {out_path}")

    # Resumen por grupo
    print("\n--- Distribución por grupo ---")
    all_keys = {k: "train" for k in train} | {k: "val" for k in val} | {k: "test" for k in test}
    group_summary = defaultdict(lambda: {"train": 0, "val": 0, "test": 0})
    for key, split in all_keys.items():
        # prefix = key.split("_linea_")[0] if "_linea_" in key else key
        prefix = None
        for prefix_i in prefixs:
            if key.startswith(prefix_i):
                prefix = prefix_i
                break

        group_summary[prefix][split] += 1

    for prefix, counts in sorted(group_summary.items()):
        total = sum(counts.values())
        print(f"  {prefix[:16]}...  total={total}  train={counts['train']}  val={counts['val']}  test={counts['test']}")

split_stratified('lauri_binarizado/split/labels_n.json', train_ratio=0.9, val_ratio=0.1, test_ratio=0.0)

In [ ]:
import json
import os

# CONFIGURACIÓN - Cambia estas rutas según tu caso
RUTA_JSON = "pinterest/seg/content.json"  # Ruta a tu archivo JSON
CARPETA_IMAGENES = "pinterest/seg"  # Carpeta donde están las imágenes

# Cargar el JSON
with open(RUTA_JSON, 'r', encoding='utf-8') as archivo:
    datos = json.load(archivo)

# Obtener todas las imágenes de la carpeta
imagenes_en_carpeta = os.listdir(CARPETA_IMAGENES)

# Verificar cuáles imágenes NO están en el JSON
for imagen in imagenes_en_carpeta:
    if imagen not in datos:
        print(f"❌ Imagen en carpeta pero no en JSON: {imagen}")

In [ ]:
import json
import os

# CONFIGURACIÓN - Cambia estas rutas según tu caso
RUTA_JSON = "pinterest/seg/content.json"  # Ruta a tu archivo JSON
CARPETA_IMAGENES = "pinterest/seg"  # Carpeta donde están las imágenes

# Cargar JSON
with open(RUTA_JSON, 'r', encoding='utf-8') as f:
    datos = json.load(f)

# Obtener archivos de la carpeta
archivos_existentes = set(os.listdir(CARPETA_IMAGENES))

# Verificar cada imagen
for nombre_imagen in datos.keys():
    if nombre_imagen not in archivos_existentes:
        print(f"❌ Imagen no encontrada: {nombre_imagen}")

print("\n✅ Verificación completada")

In [ ]:
from doctr.io import DocumentFile
from doctr.models import detection_predictor
import cv2
import numpy as np

def segment_words_doctr(image_path):
    model = detection_predictor(
        arch='db_resnet50',
        pretrained=True,
        assume_straight_pages=True
    )
    
    doc = DocumentFile.from_images(image_path)
    result = model(doc)  # result es una lista de dicts
    
    # Obtener dimensiones reales de la imagen
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    
    words = []
    for page in result:  # iterar la lista directamente
        word_array = page['words']  # array numpy [x1, y1, x2, y2, conf]
        for word in word_array:
            x1, y1, x2, y2, conf = word
            words.append({
                'bbox': (
                    int(x1 * w), int(y1 * h),
                    int(x2 * w), int(y2 * h)
                ),
                'confidence': float(conf)
            })
    
    return words

def visualize_words(image_path, words, min_confidence=0.3):
    img = cv2.imread(image_path)
    
    for word in words:
        if word['confidence'] >= min_confidence:
            x1, y1, x2, y2 = word['bbox']
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    
    return img

In [ ]:
import matplotlib.pyplot as plt

def plot_words(image_path, words, min_confidence=0.3):
    img = visualize_words(image_path, words, min_confidence)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(15, 20))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'Palabras detectadas: {len(words)} (conf >= {min_confidence})')
    plt.tight_layout()
    plt.show()

In [ ]:
words = segment_words_doctr("original_docs/20260219_120513.jpg")

In [ ]:
print(words)

In [ ]:
# Uso
plot_words("original_docs/20260219_120513.jpg", words, min_confidence=0.3)

In [ ]:
from doctr.io import DocumentFile
from doctr.models import detection_predictor

doc = DocumentFile.from_images("original_docs/20260219_120513.jpg")
predictor = detection_predictor("db_resnet50", pretrained=True)
result = predictor(doc)

for page in result.pages:
    for block in page.blocks:
        print(block.geometry)  # coordenadas normalizadas

In [ ]:
# Técnica: Morfología + Proyección de perfiles
import cv2, numpy as np
from preprocessing_utils import preprocess_image

versions = preprocess_image("original_docs/20260219_120513.jpg")
img_color = img = cv2.imread("original_docs/20260219_120513.jpg")

# Dilatación horizontal para unir palabras en bloques
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 10))
dilated = cv2.dilate(versions[5], kernel)

contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(img_color, (x,y), (x+w, y+h), (0,255,0), 2)

In [ ]:
"""
Segmentación de COLUMNAS verticales en documentos manuscritos.
Técnica: Perfil de proyección vertical → detección de valles → separación.
"""

import cv2
import numpy as np
from pathlib import Path
from preprocessing_utils import preprocess_image

def segment_columns_opencv(
    image_path: str,
    output_path: str = "result_columns.png",

    # ── Morfología ──
    dilation_kernel_w: int = 20,   # une palabras horizontalmente (no necesita ser grande)
    dilation_kernel_h: int = 80,   # une líneas verticalmente → forma "columnas sólidas"

    # ── Detección de valles ──
    valley_threshold: float = 0.02,  # fracción de la fila: si < X% de píxeles → valle
    min_valley_width: int = 20,      # ancho mínimo del valle en píxeles para ser separador

    # ── Filtrado de columnas ──
    min_column_width_ratio: float = 0.05,  # columna debe tener al menos 5% del ancho total
):
    """
    Detecta columnas de texto separándolas verticalmente.

    Flujo:
      1. Binarización → dilatación vertical fuerte → "manchas" por columna
      2. Perfil de proyección vertical (suma de píxeles por columna)
      3. Detectar valles en el perfil → posiciones de separación
      4. Recortar y guardar cada columna
    """
    img_bgr = cv2.imread(image_path)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h_img, w_img = img_gray.shape

    versions = preprocess_image(image_path=image_path)

    # ── 2. Dilatación: kernel muy alto para fusionar líneas en columnas ──
    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT, (dilation_kernel_w, dilation_kernel_h)
    )
    dilated = cv2.dilate(versions[5], kernel, iterations=2)

    # ── 3. Perfil de proyección vertical ──
    # Suma de píxeles blancos por cada columna x → array de forma (w_img,)
    projection = dilated.sum(axis=0).astype(float)          # suma por columna
    projection_norm = projection / projection.max()         # normalizar a [0, 1]

    # ── 4. Detectar valles (zonas sin tinta = separadores de columna) ──
    is_valley = projection_norm < valley_threshold           # bool array

    # Agrupar píxeles consecutivos de valle en segmentos
    valleys = []
    in_valley = False
    start = 0
    for x, val in enumerate(is_valley):
        if val and not in_valley:
            start = x
            in_valley = True
        elif not val and in_valley:
            width = x - start
            if width >= min_valley_width:
                center = (start + x) // 2
                valleys.append(center)
            in_valley = False
    if in_valley and (w_img - start) >= min_valley_width:
        valleys.append((start + w_img) // 2)

    # ── 5. Construir límites de columnas desde los valles ──
    # Añadir bordes de imagen como límites iniciales/finales
    boundaries = [0] + valleys + [w_img]

    columns = []
    for i in range(len(boundaries) - 1):
        x_start = boundaries[i]
        x_end   = boundaries[i + 1]
        width   = x_end - x_start
        if width / w_img >= min_column_width_ratio:
            columns.append((x_start, x_end))

    # ── 6. Visualización ──
    result = img_bgr.copy()
    colors = [
        (0, 200, 0),    # verde
        (0, 100, 255),  # azul
        (255, 140, 0),  # naranja
        (180, 0, 255),  # violeta
        (0, 200, 200),  # cyan
    ]

    for i, (x0, x1) in enumerate(columns):
        color = colors[i % len(colors)]
        # Rectángulo full-height para cada columna
        cv2.rectangle(result, (x0, 0), (x1, h_img), color, 3)
        # Línea de separación central
        if i > 0:
            sep_x = boundaries[i]
            cv2.line(result, (sep_x, 0), (sep_x, h_img), (0, 0, 255), 2)
        label = f"Col {i+1}"
        cv2.putText(result, label, (x0 + 5, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    cv2.imwrite(output_path, result)

    # ── 7. Guardar también el perfil de proyección como imagen de diagnóstico ──
    _save_projection_plot(projection_norm, valleys, output_path, w_img)

    print(f"[✓] {len(columns)} columna(s) detectada(s): {output_path}")
    for i, (x0, x1) in enumerate(columns):
        print(f"    Col {i+1}: x=[{x0} → {x1}]  ancho={x1-x0}px")

    return columns   # lista de (x_start, x_end)

def _save_projection_plot(projection_norm, valleys, output_path, w_img):
    """Genera imagen de diagnóstico con el perfil de proyección y los valles."""
    plot_h = 150
    plot = np.ones((plot_h, w_img, 3), dtype=np.uint8) * 240

    # Dibujar perfil
    for x in range(w_img - 1):
        y1 = int((1 - projection_norm[x])     * (plot_h - 10)) + 5
        y2 = int((1 - projection_norm[x + 1]) * (plot_h - 10)) + 5
        cv2.line(plot, (x, y1), (x + 1, y2), (30, 30, 200), 1)

    # Marcar valles detectados
    for vx in valleys:
        cv2.line(plot, (vx, 0), (vx, plot_h), (0, 0, 200), 2)

    diag_path = output_path.replace(".png", "_projection.png")
    cv2.imwrite(diag_path, plot)
    print(f"[✓] Perfil de proyección: {diag_path}")

def extract_column_crops(image_path: str, columns: list, output_dir: str = "."):
    """
    Opcional: recorta y guarda cada columna como imagen independiente.
    Útil si luego quieres procesar cada columna por separado (ej: con DocTR).
    """
    img_bgr = cv2.imread(image_path)
    h_img = img_bgr.shape[0]
    out = Path(output_dir)

    crops = []
    for i, (x0, x1) in enumerate(columns):
        crop = img_bgr[0:h_img, x0:x1]
        crop_path = str(out / f"column_{i+1}.png")
        cv2.imwrite(crop_path, crop)
        crops.append(crop_path)
        print(f"[✓] Columna {i+1} guardada: {crop_path}")

    return crops

In [ ]:
IMAGE_PATH = "original_docs/20260219_120320.jpg"

columns = segment_columns_opencv(
    IMAGE_PATH,
    output_path="resultado_columnas.png",

    # ── Ajustar según tu imagen ──
    dilation_kernel_w=20,       # no necesita ser grande, solo une palabras
    dilation_kernel_h=80,       # grande: fusiona líneas verticalmente en columnas
    valley_threshold=0.02,      # bajar si las columnas se tocan; subir si hay mucho ruido
    min_valley_width=10,        # ancho mínimo del espacio entre columnas
    min_column_width_ratio=0.05 # ignora columnas más pequeñas que el 5% del ancho
)

# Opcional: guardar cada columna como imagen separada
# extract_column_crops(IMAGE_PATH, columns, output_dir=".")

In [ ]:
import pandas as pd
from PIL import Image
import io
import matplotlib.pyplot as plt

df = pd.read_parquet("IAM/data/train.parquet")

# Las imágenes están como dict con key 'bytes'
# Función para decodificar
def bytes_to_image(image_field):
    img_bytes = image_field['bytes']
    return Image.open(io.BytesIO(img_bytes))

print(len(df['image']))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    img = bytes_to_image(df['image'][i])
    ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
    ax.set_title(df['text'][i][:30], fontsize=8, wrap=True)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Tamaños de todas las imágenes
sizes = [bytes_to_image(row).size for row in df['image']]
widths  = [s[0] for s in sizes]
heights = [s[1] for s in sizes]

print("Ancho  - min:", min(widths),  "max:", max(widths),  "promedio:", sum(widths)//len(widths))
print("Alto   - min:", min(heights), "max:", max(heights), "promedio:", sum(heights)//len(heights))

# Modos (RGB, L, RGBA...)
modes = set(bytes_to_image(row).mode for row in df['image'])
print("Modos encontrados:", modes)

# Distribución de textos por longitud
df['text_len'] = df['text'].str.len()
print(df['text_len'].describe())

In [ ]:
"""
extract_parquet.py
──────────────────
Extrae todas las imágenes de un archivo .parquet y las guarda en una carpeta,
generando un JSON con la estructura:
    { "nombre_imagen.jpg": "transcripción", ... }
 
Uso
---
    python extract_parquet.py --parquet ruta/al/archivo.parquet --output carpeta_salida
 
    # Con columnas personalizadas:
    python extract_parquet.py --parquet data.parquet --output salida \
        --image-col image --text-col text --format png --prefix sample
"""
 
import argparse
import json
import os
from pathlib import Path
 
import pandas as pd
from PIL import Image
from tqdm import tqdm
 
def extract_parquet(
    parquet_path: str,
    output_dir: str,
    image_col: str = "image",
    text_col: str = "text",
    img_format: str = "jpg",
    prefix: str = "img",
):
    """
    Parámetros
    ----------
    parquet_path : ruta al archivo .parquet
    output_dir   : carpeta donde se guardarán las imágenes y el JSON
    image_col    : nombre de la columna que contiene las imágenes PIL
    text_col     : nombre de la columna con las transcripciones
    img_format   : formato de salida de las imágenes ('jpg' o 'png')
    prefix       : prefijo del nombre de archivo  →  {prefix}_{idx:06d}.{format}
    """
    parquet_path = Path(parquet_path)
    output_dir   = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
 
    img_format = img_format.lower().strip(".")
    pil_format = "JPEG" if img_format == "jpg" else img_format.upper()
 
    print(f"Leyendo parquet : {parquet_path}")
    df = pd.read_parquet(parquet_path)
    print(f"Filas encontradas: {len(df)}")
 
    # Validar columnas
    missing = [c for c in (image_col, text_col) if c not in df.columns]
    if missing:
        raise ValueError(
            f"Columnas no encontradas: {missing}. "
            f"Columnas disponibles: {list(df.columns)}"
        )
 
    transcriptions = {}
    errors          = []
 
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extrayendo"):
        filename = f"{prefix}_{idx:06d}.{img_format}"
        img_path = output_dir / filename
 
        try:
            img = row[image_col]
 
            # Formato HuggingFace en parquet crudo:
            # {'bytes': b'\xff\xd8...', 'path': 'archivo.jpg'}
            if isinstance(img, dict):
                import io
                raw = img.get("bytes") or img.get("data")
                if raw:
                    img = Image.open(io.BytesIO(raw))
                else:
                    # Fallback: intentar abrir por path si viene en el dict
                    fallback_path = img.get("path")
                    if fallback_path and Path(fallback_path).exists():
                        img = Image.open(fallback_path)
                    else:
                        raise ValueError(f"Dict de imagen sin 'bytes' ni 'path' válido: {list(img.keys())}")
 
            # Si viene directamente como bytes planos
            elif isinstance(img, bytes):
                import io
                img = Image.open(io.BytesIO(img))
 
            # Convertir a RGB para JPEG (no admite RGBA o P)
            if pil_format == "JPEG" and img.mode not in ("RGB", "L"):
                img = img.convert("RGB")
 
            img.save(img_path, format=pil_format)
            transcriptions[filename] = str(row[text_col])
 
        except Exception as e:
            errors.append({"index": idx, "error": str(e)})
            print(f"\n  ⚠ Error en fila {idx}: {e} — omitida")
 
    # Guardar JSON
    json_path = output_dir / "transcriptions.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(transcriptions, f, ensure_ascii=False, indent=2)
 
    # Guardar log de errores si los hay
    if errors:
        errors_path = output_dir / "errors.json"
        with open(errors_path, "w", encoding="utf-8") as f:
            json.dump(errors, f, ensure_ascii=False, indent=2)
        print(f"\n⚠ {len(errors)} errores guardados en: {errors_path}")
 
    total_ok = len(transcriptions)
    print(f"\n✓ {total_ok} imágenes guardadas en   : {output_dir}")
    print(f"✓ Transcripciones guardadas en       : {json_path}")
    print(f"  Errores                            : {len(errors)}")
 
    return transcriptions

In [ ]:
extract_parquet('IAM/data/train.parquet', 'IAM_img/train')

In [ ]:
extract_parquet('IAM/data/validation.parquet', 'IAM_img/val')

In [ ]:
extract_parquet('IAM/data/test.parquet', 'IAM_img/test')

In [ ]:
import cv2
import numpy as np
from scipy.ndimage import distance_transform_edt, uniform_filter1d
from scipy.signal import find_peaks
from skimage.filters import threshold_sauvola
import os
from preprocessing_utils import _binarize

# ─── Segmentación por líneas: HPP + Seam Carving con SDT ──────────────────────

def compute_sdt_energy(binary_ink: np.ndarray) -> np.ndarray:
    """
    Signed Distance Transform (Saabni & El-Sana, 2011 / Das & Panda, 2023).

    Mapa de energía:
      - Píxeles DENTRO de tinta  → energía NEGATIVA (alto coste → seam los evita)
      - Píxeles FUERA de tinta   → energía POSITIVA (bajo coste → seam pasa por aquí)

    El seam de mínima energía atravesará los espacios entre líneas.
    """
    ink_mask = binary_ink < 128          # True donde hay tinta (fondo negro = tinta)
    dist_outside = distance_transform_edt(~ink_mask).astype(np.float32)  # dist al ink desde bg
    dist_inside  = distance_transform_edt( ink_mask).astype(np.float32)  # dist al bg desde ink
    return dist_outside - dist_inside    # positivo en background, negativo en tinta

def find_seam_dp(energy: np.ndarray, seed_row: int, half_band: int) -> np.ndarray:
    """
    Programación dinámica para seam horizontal de mínima energía.

    Restricciones:
      - Empieza cerca de seed_row (ventana ±half_band)
      - Se desplaza máximo ±1 fila por columna (garantiza continuidad)
      - Nunca sale de la banda vertical [r_min, r_max]

    Returns: array (W,) con la fila del seam para cada columna.
    """
    H, W = energy.shape
    r_min = max(0, seed_row - half_band)
    r_max = min(H, seed_row + half_band)
    band_h = r_max - r_min
    if band_h < 1:
        return np.full(W, seed_row, dtype=np.int32)

    E = energy[r_min:r_max, :].copy()   # subregión

    # Tabla DP: dp[r, c] = mínima energía acumulada para llegar a (r, c)
    dp   = np.full((band_h, W), np.inf, dtype=np.float32)
    from_  = np.zeros((band_h, W), dtype=np.int8)  # desplazamiento del predecesor

    dp[:, 0] = E[:, 0]

    for c in range(1, W):
        for r in range(band_h):
            best_cost = np.inf
            best_dr   = 0
            for dr in (-1, 0, 1):
                pr = r + dr
                if 0 <= pr < band_h and dp[pr, c-1] < best_cost:
                    best_cost = dp[pr, c-1]
                    best_dr   = dr
            dp[r, c]   = E[r, c] + best_cost
            from_[r, c] = best_dr

    # Retroceso desde la columna final
    seam_local = np.zeros(W, dtype=np.int32)
    seam_local[-1] = int(np.argmin(dp[:, -1]))
    for c in range(W - 2, -1, -1):
        seam_local[c] = seam_local[c+1] - from_[seam_local[c+1], c+1]
        seam_local[c] = np.clip(seam_local[c], 0, band_h - 1)

    return seam_local + r_min   # devuelve coordenadas absolutas

def hpp_seed_rows(binary_ink: np.ndarray,
                  min_peak_height_ratio: float = 0.05,#0.05
                  min_distance_px: int = 55,
                  prominence_ratio: float = 0.10) -> np.ndarray:
    """
    Detección de filas semilla usando Horizontal Projection Profile.

    Cada pico corresponde al centroide vertical de una línea de texto.
    Los valles entre picos serán los puntos de inicio de los seams.
    """
    ink_proj = np.sum(binary_ink < 128, axis=1).astype(float)
    ink_smooth = uniform_filter1d(ink_proj, size=7)

    max_val = ink_smooth.max()
    peaks, _ = find_peaks(
        ink_smooth,
        height     = max_val * min_peak_height_ratio,
        distance   = min_distance_px,
        prominence = max_val * prominence_ratio,
    )
    return peaks

def valley_between(ink_proj_smooth: np.ndarray, r1: int, r2: int) -> int:
    """Fila de mínimo valor de proyección entre dos picos → candidato a seam."""
    segment = ink_proj_smooth[r1:r2]
    return r1 + int(np.argmin(segment))

def segment_lines_seam_carving(
    image_path: str,
    output_dir: str,
    half_band: int = 18,
    pad_px: int    = 12, #6
    debug: bool    = False,
) -> list[str]:
    """
    Pipeline completo:
      1. Lee imagen en escala de grises
      2. Binariza (Sauvola + corrección de fondo)
      3. Calcula mapa de energía SDT
      4. HPP → picos de líneas → valles como semillas de seams
      5. Seam Carving (DP) por cada intervalo entre líneas
      6. Recorta y guarda cada franja como JPG

    Parameters
    ----------
    image_path : ruta a la imagen original (color o gris)
    output_dir : carpeta de salida
    half_band  : radio en px de la banda de búsqueda del seam alrededor de la semilla
    pad_px     : padding vertical extra en cada recorte
    debug      : si True, guarda imagen con los seams dibujados

    Returns
    -------
    Lista de rutas a los JPGs generados
    """
    os.makedirs(output_dir, exist_ok=True)

    # 1. Carga
    img_bgr = cv2.imread(image_path)
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    H, W    = gray.shape

    # 2. Binarización (tu pipeline)
    binary = _binarize(gray)              # 255 = fondo claro, 0/oscuro = tinta

    # 3. Mapa de energía SDT
    energy = compute_sdt_energy(binary)

    # 4. HPP → picos y valles semilla
    ink_proj  = np.sum(binary < 128, axis=1).astype(float)
    ink_smooth = uniform_filter1d(ink_proj, size=7)
    peaks = hpp_seed_rows(binary)

    if len(peaks) < 2:
        # Sin suficientes picos: recorte único
        out_path = os.path.join(output_dir, "linea_01.jpg")
        cv2.imwrite(out_path, img_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
        return [out_path]

    # Calcular los seams separadores entre cada par de líneas consecutivas
    # Seam 0     → antes de la primera línea  (borde superior)
    # Seam 1..N  → entre líneas
    # Seam N+1   → después de la última línea (borde inferior)
    seed_rows = []

    # Borde superior: valle antes del primer pico
    seed_rows.append(valley_between(ink_smooth, 0, peaks[0]) if peaks[0] > 0 else 0)

    # Valles entre picos consecutivos
    for i in range(len(peaks) - 1):
        seed_rows.append(valley_between(ink_smooth, peaks[i], peaks[i+1]))

    # Borde inferior: valle después del último pico
    seed_rows.append(
        valley_between(ink_smooth, peaks[-1], H) if peaks[-1] < H - 1 else H - 1
    )

    # Calcular seams con DP
    seams = []
    for seed in seed_rows:
        seam = find_seam_dp(energy, seed_row=seed, half_band=half_band)
        seams.append(seam)

    # 5. Recortar entre seams consecutivos
    saved_paths = []
    for i in range(len(seams) - 1):
        top_seam    = seams[i]
        bottom_seam = seams[i + 1]

        y1 = max(0,  int(np.percentile(top_seam,    90)) - pad_px)
        y2 = min(H,  int(np.percentile(bottom_seam, 10)) + pad_px)

        # y1 = max(0, int(top_seam.min())    - pad_px)   # .min() en vez de .max()
        # y2 = min(H, int(bottom_seam.max()) + pad_px)   # .max() en vez de .min()

        if y2 - y1 < 5:
            continue

        crop = img_bgr[y1:y2, 0:W]
        out_path = os.path.join(output_dir, f"fe39e3c8686d2b89a40990e917f1844c_linea_{i+1:02d}.jpg")
        cv2.imwrite(out_path, crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
        saved_paths.append(out_path)
        print(f"  Guardada: {out_path}  (y={y1}–{y2})")

    # 6. (Opcional) Visualización de debug: dibuja seams sobre la imagen
    if debug:
        debug_img = img_bgr.copy()
        for seam in seams:
            for col, row in enumerate(seam):
                if 0 <= row < H:
                    cv2.circle(debug_img, (col, row), 0, (0, 0, 255), -1)
        debug_path = os.path.join(output_dir, "_debug_seams.jpg")
        cv2.imwrite(debug_path, debug_img)
        print(f"  Debug guardado: {debug_path}")

    return saved_paths

rutas = segment_lines_seam_carving(
        image_path = "pinterest/segment/fe39e3c8686d2b89a40990e917f1844c.jpg",
        output_dir = "pinterest/segment",
        half_band  = 2,   # ajustar según el interlineado del documento
        pad_px     = 2,
        debug      = True, # activa imagen con seams dibujados en rojo
    )
print(f"\nTotal líneas segmentadas: {len(rutas)}")

In [ ]:
"""
vertical_crop.py
-----------------
Dado una imagen de línea de texto (binarizada o en gris),
recorta el espacio blanco de los lados izquierdo y derecho,
dejando un margen configurable para que el texto no quede encogido.

Uso individual:
    python vertical_crop.py --input imagen.png --output recortada.png

Uso sobre carpeta completa:
    python vertical_crop.py --input carpeta/ --output carpeta_recortada/ --margin 20

Parámetros:
    --margin   : píxeles de margen a dejar a cada lado (default: 20)
    --threshold: umbral para considerar píxel como "tinta" (0-255, default: 200)
                 píxeles < threshold se consideran tinta (oscuros)
"""

import argparse
import sys
from pathlib import Path

import cv2
import numpy as np

SUPPORTED = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def find_text_columns(gray: np.ndarray, ink_threshold: int = 200) -> tuple[int, int]:
    """
    Encuentra la columna más a la izquierda y más a la derecha
    donde hay píxeles de tinta (oscuros).

    Retorna (col_izq, col_der) — índices de columna inclusivos.
    """
    # Máscara booleana: True donde hay tinta
    ink_mask = gray < ink_threshold          # shape (H, W)
    cols_with_ink = np.where(ink_mask.any(axis=0))[0]

    if len(cols_with_ink) == 0:
        # Imagen totalmente blanca — devolver bordes originales
        return 0, gray.shape[1] - 1

    return int(cols_with_ink[0]), int(cols_with_ink[-1])

def crop_horizontal(image: np.ndarray,
                    margin: int = 20,
                    ink_threshold: int = 200,
                    fill_value: int = 255) -> np.ndarray:
    """
    Recorta el espacio blanco lateral de una imagen de línea de texto.

    Args:
        image        : array uint8 en escala de grises
        margin       : píxeles de margen a preservar a cada lado
        ink_threshold: umbral para detectar tinta (píxeles < threshold = tinta)
        fill_value   : color de relleno si el margen se sale del borde (255 = blanco)

    Returns:
        Imagen recortada con el margen aplicado.
    """
    h, w = image.shape
    col_izq, col_der = find_text_columns(image, ink_threshold)

    # Aplicar margen y clampar a los bordes reales
    x1 = max(0, col_izq - margin)
    x2 = min(w - 1, col_der + margin)

    cropped = image[:, x1:x2 + 1]

    # Si el margen pedido era mayor que el espacio disponible,
    # rellenar con blanco para no mentir al modelo
    pad_left  = max(0, margin - col_izq)
    pad_right = max(0, (col_der + margin) - (w - 1))

    if pad_left > 0 or pad_right > 0:
        cropped = np.pad(
            cropped,
            pad_width=((0, 0), (pad_left, pad_right)),
            mode='constant',
            constant_values=fill_value
        )

    return cropped

def process_single(input_path: Path, output_path: Path,
                   margin: int, threshold: int) -> bool:
    gray = cv2.imread(str(input_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        print(f"  ⚠ No se pudo leer: {input_path.name}")
        return False

    cropped = crop_horizontal(gray, margin=margin, ink_threshold=threshold)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    ok = cv2.imwrite(str(output_path), cropped)
    if not ok:
        print(f"  ⚠ No se pudo guardar: {output_path}")
        return False

    orig_w   = gray.shape[1]
    crop_w   = cropped.shape[1]
    saved_px = orig_w - crop_w
    return True

def process_folder(input_dir: Path, output_dir: Path,
                   margin: int, threshold: int, max_html: int):
    output_dir.mkdir(parents=True, exist_ok=True)

    images = sorted([
        p for p in input_dir.rglob('*')
        if p.suffix.lower() in SUPPORTED
    ])

    if not images:
        print(f"No se encontraron imágenes en: {input_dir}")
        sys.exit(1)

    print(f"Imágenes encontradas : {len(images)}")
    print(f"Margen               : {margin} px")
    print(f"Umbral de tinta      : < {threshold}")
    print(f"Guardando en         : {output_dir}\n")

    stats = []
    for i, img_path in enumerate(images):
        gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            print(f"  ⚠ No se pudo leer: {img_path.name}")
            stats.append((img_path, None, None, False))
            continue

        col_izq, col_der = find_text_columns(gray, threshold)
        cropped          = crop_horizontal(gray, margin, threshold)

        rel      = img_path.relative_to(input_dir)
        out_path = output_dir / rel.with_suffix('.png')
        out_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_path), cropped)

        orig_w = gray.shape[1]
        crop_w = cropped.shape[1]
        stats.append((img_path, orig_w, crop_w, True))

        if (i + 1) % 100 == 0 or (i + 1) == len(images):
            reduccion = 100 * (1 - crop_w / orig_w) if orig_w else 0
            print(f"  [{i+1}/{len(images)}] {img_path.name}  "
                  f"{orig_w}→{crop_w}px  (-{reduccion:.0f}%)")

    ok_count    = sum(ok for *_, ok in stats)
    valid_stats = [(orig, crop) for _, orig, crop, ok in stats if ok]
    if valid_stats:
        reducciones = [100 * (1 - c / o) for o, c in valid_stats if o]
        print(f"\n── Resumen ──")
        print(f"  Procesadas    : {ok_count}/{len(images)}")
        print(f"  Reducción media de ancho: {np.mean(reducciones):.1f}%")
        print(f"  Reducción máxima        : {max(reducciones):.1f}%")

    # HTML de comparativa
    html_path = output_dir / "_comparativa_crop.html"
    _generate_html(input_dir, output_dir, images, stats, html_path, max_html)
    print(f"\n✓ HTML de comparativa : {html_path}")
    print(f"✓ Listo.")

def _generate_html(input_dir, output_dir, images, stats, html_path, max_html):
    items = []
    for img_path, orig_w, crop_w, ok in zip(images[:max_html], stats[:max_html]):
        img_path, orig_w, crop_w, ok = img_path, *stats[images.index(img_path)]

        rel     = img_path.relative_to(input_dir)
        out_p   = output_dir / rel.with_suffix('.png')
        orig_uri = img_path.resolve().as_uri()
        crop_uri = out_p.resolve().as_uri() if ok else ""

        reduccion_str = ""
        if orig_w and crop_w:
            r = 100 * (1 - crop_w / orig_w)
            reduccion_str = f"  {orig_w}→{crop_w}px (-{r:.0f}%)"

        crop_tag = (f'<img src="{crop_uri}" alt="recortada">'
                    if ok else '<p class="error">ERROR</p>')

        items.append(f"""
        <div class="card">
          <p class="fname">{img_path.name}{reduccion_str}</p>
          <div class="pair">
            <div><p>Original</p><img src="{orig_uri}"></div>
            <div><p>Recortada</p>{crop_tag}</div>
          </div>
        </div>""")

    note = ""
    total = len(images)
    if total > max_html:
        note = f"<p style='color:#888'>Mostrando {max_html} de {total}.</p>"

    html = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <title>Inspección de recorte lateral</title>
  <style>
    body   {{ font-family: sans-serif; background: #1a1a1a; color: #eee; padding: 20px; }}
    h1     {{ color: #fff; }}
    .card  {{ border: 1px solid #444; border-radius: 8px; padding: 12px;
              margin-bottom: 20px; background: #2a2a2a; }}
    .fname {{ font-size: 0.85em; color: #aaa; margin: 0 0 8px; }}
    .pair  {{ display: flex; gap: 16px; flex-wrap: wrap; }}
    .pair > div {{ flex: 1; min-width: 200px; }}
    .pair p {{ font-size: 0.8em; color: #888; margin: 0 0 4px; }}
    img    {{ max-width: 100%; border: 1px solid #555; border-radius: 4px;
              background: white; image-rendering: pixelated; }}
    .error {{ color: #ff6b6b; }}
  </style>
</head>
<body>
  <h1>Inspección de recorte lateral</h1>
  {note}
  {"".join(items)}
</body>
</html>"""

    html_path.write_text(html, encoding='utf-8')

# process_single(Path('0.png'), Path('0_0.png'), 20, 200)
process_folder(Path('binar/test'), Path('binar/test/segm'), 20, 200, 200)

In [ ]:
import json
import os
import numpy as np
from scipy.stats import wilcoxon
from collections import defaultdict
from hcr import HTRInference
import torch

CHARSET = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
               "0123456789 $#&%*.,;:¡!¿?«»-_'\"()[]áéíóúüñÁÉÍÓÚÜÑ")
IMG_HEIGHT = 64          # altura de precomputación y entrada al modelo
IMG_WIDTH  = 256         # FIX 3: era 320 → 256  (suficiente para palabras)
BATCH_SIZE = 128         # FIX 5: era 96 → 128   (mejor saturación GPU con AMP)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def edit_distance(s1, s2):
    """Distancia de Levenshtein entre dos secuencias."""
    m, n = len(s1), len(s2)
    dp = np.zeros((m + 1, n + 1), dtype=int)

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j],    # borrar
                                       dp[i][j-1],    # insertar
                                       dp[i-1][j-1])  # sustituir
    return dp[m][n]

IMAGE_FOLDER = "IAM_img/test"
JSON_PATH    = "IAM_img/test/transcriptions.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    el_json = json.load(f)

inferencer_best = HTRInference(
    model_path='checkpoints/ver4_from0/best_model_2.pt',       # época 28
    charset=CHARSET, device=DEVICE,
    img_height=IMG_HEIGHT, img_width=IMG_WIDTH,
    hidden_channels=32, lstm_hidden=128,
)

inferencer_early = HTRInference(
    model_path='checkpoints/ver4_from0/checkpoint_epoch010_cer0.0989.pt',      # época ~4-6
    charset=CHARSET, device=DEVICE,
    img_height=IMG_HEIGHT, img_width=IMG_WIDTH,
    hidden_channels=32, lstm_hidden=128,
)

image_entries = []
for entry in sorted(os.scandir(IMAGE_FOLDER), key=lambda e: e.name):
    if entry.is_file():
        image_entries.append((entry.name, entry.path))
    elif entry.is_dir():
        for sub in sorted(os.scandir(entry.path), key=lambda e: e.name):
            if sub.is_file():
                image_entries.append((f"{entry.name}/{sub.name}", sub.path))

refs        = []
hyps_best   = []
hyps_early  = []

print("Evaluando ambos modelos...\n")

for key_ext, image_path in image_entries:
    key_no_ext = os.path.splitext(key_ext)[0]
    clave = key_ext if key_ext in el_json else (key_no_ext if key_no_ext in el_json else None)
    if clave is None:
        continue

    reference = el_json[clave]
    pred_best  = inferencer_best.predict(image_path)
    pred_early = inferencer_early.predict(image_path)

    refs.append(reference)
    hyps_best.append(pred_best)
    hyps_early.append(pred_early)

def cer_global(preds, targets):
    total_edit = sum(edit_distance(p, t) for p, t in zip(preds, targets))
    total_chars = sum(len(t) for t in targets)
    return total_edit / max(total_chars, 1)

def wer_global(preds, targets):
    total_edit = sum(edit_distance(p.split(), t.split()) for p, t in zip(preds, targets))
    total_words = sum(len(t.split()) for t in targets)
    return total_edit / max(total_words, 1)

def cer_por_muestra(preds, targets):
    return np.array([
        edit_distance(p, t) / max(len(t), 1)
        for p, t in zip(preds, targets)
    ])

cer_arr_best  = cer_por_muestra(hyps_best,  refs)
cer_arr_early = cer_por_muestra(hyps_early, refs)

diffs = cer_arr_best - cer_arr_early
if np.all(diffs == 0):
    stat, p_value = None, 1.0
else:
    stat, p_value = wilcoxon(cer_arr_best, cer_arr_early)

n = len(refs)
print("═" * 55)
print("  COMPARACIÓN DE MODELOS — TEST SET IAM")
print("═" * 55)
print(f"  Muestras evaluadas  : {n}")
print()
print(f"  {'Métrica':<22} {'best_model':>12} {'early_model':>12}")
print(f"  {'─'*46}")
print(f"  {'CER global':<22} {cer_global(hyps_best, refs):>11.4f}  "
      f"{cer_global(hyps_early, refs):>11.4f}")
print(f"  {'WER global':<22} {wer_global(hyps_best, refs):>11.4f}  "
      f"{wer_global(hyps_early, refs):>11.4f}")
print(f"  {'CER medio ± std':<22} "
      f"{cer_arr_best.mean():>6.4f}±{cer_arr_best.std():.4f}  "
      f"{cer_arr_early.mean():>6.4f}±{cer_arr_early.std():.4f}")
print(f"  {'CER mediana':<22} {np.median(cer_arr_best):>11.4f}  "
      f"{np.median(cer_arr_early):>11.4f}")
print()
print(f"  Wilcoxon  stat={stat}  p={p_value:.4f}")
print()

ALPHA = 0.05
if p_value < ALPHA:
    ganador = "best_model" if cer_arr_best.mean() < cer_arr_early.mean() else "early_model"
    print(f"  → Diferencia SIGNIFICATIVA (p<{ALPHA})")
    print(f"  → Usar: {ganador}")
else:
    # Sin significancia estadística: elegir el de menor CER medio como desempate
    ganador = "best_model" if cer_arr_best.mean() <= cer_arr_early.mean() else "early_model"
    print(f"  → Sin diferencia significativa (p≥{ALPHA})")
    print(f"  → Desempate por CER medio: usar {ganador}")

print("═" * 55)